# Control-aware feature selection

This notebook defines the supervisor-requested feature recipe: strong, interval-consistent FRDA change is prioritised while same-direction change in controls is penalised. It uses exactly the same 70 candidates and outer folds as the FRDA-only baseline.

Only outer-training controls can affect a fold's ranking. Held-out controls remain untouched until downstream model evaluation.


## 1. Shared experiment contract and cohorts

The persisted fold assignment makes the comparison paired and reproducible. This notebook fails if the dataset, feature panel, or folds no longer match the baseline contract.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.trackfa_pairs import trackfa_pairs_to_long
from src.eval.control_aware_selection import wide_cohort_to_pair_long
from src.eval.feature_recipes import (
    build_fold_local_feature_recipe,
    build_full_data_feature_recipe,
    compare_feature_recipes,
    feature_recipe_stability,
)
from src.features.panels import a_priori_70_feature_names
from src.reporting.experiment_artifacts import (
    build_experiment_manifest,
    make_participant_folds,
    read_experiment_contract,
    read_table_artifact,
    write_experiment_contract,
    write_table_artifact,
)

repo_data_root = REPO_ROOT / "data" / "processed"
fallback_data_root = Path("/Users/robertwang/Documents/New_project/biomarkers/data/processed")
DATA_ROOT = repo_data_root if repo_data_root.is_dir() else fallback_data_root
PAIRS_PATH = DATA_ROOT / "trackfa_pairs_drop3poms.csv"
WIDE_PATH = DATA_ROOT / "trackfa_merged_wide.csv"
for required in (PAIRS_PATH, WIDE_PATH):
    if not required.is_file():
        raise FileNotFoundError(f"Required input not found: {required}")

RUN_ID = "trackfa_70_feature_comparison_v1"
RUN_DIR = REPO_ROOT / "results" / "experiments" / RUN_ID
SELECTION_DIR = RUN_DIR / "selections"
SELECTION_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
OUTER_FOLDS = 5
INNER_FOLDS = 5
FEATURE_COUNT = 16
CONTROL_PENALTY = 0.5
feature_names = a_priori_70_feature_names()
assert len(feature_names) == 70 and len(set(feature_names)) == 70

pairs = pd.read_csv(PAIRS_PATH, low_memory=False)
frda_long = trackfa_pairs_to_long(pairs)
missing_frda = [feature for feature in feature_names if feature not in frda_long.columns]
if missing_frda:
    raise KeyError(f"FRDA data are missing panel features: {missing_frda}")


manifest, folds = read_experiment_contract(RUN_DIR)
wide = pd.read_csv(WIDE_PATH, low_memory=False)
control_long = wide_cohort_to_pair_long(wide, feature_names, cohort_value=1)
control_long["cohort"] = "Control"
missing_control = [feature for feature in feature_names if feature not in control_long.columns]
if missing_control:
    raise KeyError(f"Control data are missing panel features: {missing_control}")

baseline_recipe = read_table_artifact(
    SELECTION_DIR / "frda_only_features_by_fold.csv",
    schema="feature_recipe",
    manifest=manifest,
    panel=feature_names,
)
cohort_counts = pd.DataFrame([
    {"Cohort": "FRDA", "Participants": frda_long["subject"].nunique(), "Annual pairs": frda_long["pair_id"].nunique()},
    {"Cohort": "Control", "Participants": control_long["subject"].nunique(), "Annual pairs": control_long["pair_id"].nunique()},
])
display(cohort_counts)
display(pd.DataFrame([{
    "Candidate features": len(feature_names),
    "Selected per fold": FEATURE_COUNT,
    "Control penalty": CONTROL_PENALTY,
    "Outer folds": OUTER_FOLDS,
}]))


,Cohort,Participants,Annual pairs
0,FRDA,117,207
1,Control,95,190


,Candidate features,Selected per fold,Control penalty,Outer folds
0,70,16,0.5,5


## 2. Fold-local control-aware ranking

For each feature, FRDA change is oriented to its pooled FRDA direction. The selection score retains the weaker of the two interval effects and subtracts $0.5$ times any control change in the same direction. Opposite-direction control change is not penalised.


In [2]:
control_recipe = build_fold_local_feature_recipe(
    frda_long,
    feature_names,
    folds,
    strategy="control_aware",
    k=FEATURE_COUNT,
    control_long=control_long,
    control_penalty=CONTROL_PENALTY,
)
write_table_artifact(
    SELECTION_DIR / "control_aware_features_by_fold.csv",
    control_recipe,
    schema="feature_recipe",
    manifest=manifest,
    metadata={
        "scope": "outer-training folds",
        "k": FEATURE_COUNT,
        "control_penalty": CONTROL_PENALTY,
        "held_out_controls_used": False,
    },
    panel=feature_names,
)

selected_by_fold = (
    control_recipe.loc[control_recipe["selected"]]
    .groupby("outer_fold", as_index=False)
    .agg(
        Selected_features=("feature", "size"),
        Training_FRDA_participants=("train_frda_participants", "first"),
        Training_control_participants=("train_control_participants", "first"),
        Features=("feature", lambda values: " | ".join(values)),
    )
    .rename(columns={"outer_fold": "Outer fold"})
)
display(selected_by_fold)


,Outer fold,Selected_features,Training_FRDA_participants,Training_control_participants,Features
0,1,16,93,76,Midbrain | Pons | Cerebellum_Cortex_CerebNet |...
1,2,16,93,76,Cerebellum_Cortex_CerebNet | Midbrain | Pons |...
2,3,16,94,76,Pons | Cerebellum_Cortex_CerebNet | Cerebellum...
3,4,16,94,76,Cerebellum_Cortex_CerebNet | Pons | Cerebellum...
4,5,16,94,76,Cerebellum_WM_CerebNet | Cerebellum_Cortex_Cer...


## 3. FRDA and control marginal effects

These are single-feature paired effect sizes from the training portions of each fold. They describe each feature alone; they are not conditional coefficients from a fitted composite.


In [3]:
control_frequency, control_jaccard = feature_recipe_stability(control_recipe)
effect_summary = (
    control_recipe.groupby("feature", as_index=False)
    .agg(
        Mean_rank=("rank", "mean"),
        Mean_selection_score=("selection_score", "mean"),
        FRDA_pooled_d=("frda_pooled_d_z", "mean"),
        FRDA_V1_V2_d=("frda_v1_v2_d_z", "mean"),
        FRDA_V2_V3_d=("frda_v2_v3_d_z", "mean"),
        Control_pooled_d=("control_pooled_d_z", "mean"),
        Control_V1_V2_d=("control_v1_v2_d_z", "mean"),
        Control_V2_V3_d=("control_v2_v3_d_z", "mean"),
    )
    .merge(control_frequency[["feature", "selection_frequency"]], on="feature", how="left")
    .fillna({"selection_frequency": 0.0})
    .sort_values(["selection_frequency", "Mean_rank"], ascending=[False, True], kind="mergesort")
)
effect_summary.insert(0, "Rank", np.arange(1, len(effect_summary) + 1))
effect_summary = effect_summary.rename(columns={"feature": "Feature", "selection_frequency": "Selection frequency"})
display(effect_summary.head(30).round(3))

stability_summary = pd.DataFrame([{
    "Strategy": "Control-aware",
    "Outer folds": OUTER_FOLDS,
    "Selected per fold": FEATURE_COUNT,
    "Mean Jaccard": control_jaccard["jaccard"].mean(),
    "Minimum Jaccard": control_jaccard["jaccard"].min(),
    "Maximum Jaccard": control_jaccard["jaccard"].max(),
}])
display(stability_summary.round(3))
control_frequency.to_csv(SELECTION_DIR / "control_aware_selection_frequency.csv", index=False)
control_jaccard.to_csv(SELECTION_DIR / "control_aware_selection_jaccard.csv", index=False)


,Rank,Feature,Mean_rank,Mean_selection_score,FRDA_pooled_d,FRDA_V1_V2_d,FRDA_V2_V3_d,Control_pooled_d,Control_V1_V2_d,Control_V2_V3_d,Selection frequency
1,1,Cerebellum_Cortex_CerebNet,1.8,0.392,-0.621,-0.881,-0.403,-0.020,-0.169,0.158,1.0
34,2,Pons,2.2,0.359,-0.552,-0.778,-0.359,0.509,0.412,0.622,1.0
2,3,Cerebellum_WM_CerebNet,3.0,0.348,-0.426,-0.512,-0.348,0.229,0.320,0.149,1.0
32,4,Midbrain,3.2,0.322,-0.353,-0.389,-0.325,0.327,0.303,0.352,1.0
64,5,Thalamus,5.6,0.249,-0.369,-0.475,-0.249,0.090,0.028,0.156,1.0
18,6,FA_PTR,6.6,0.208,-0.254,-0.208,-0.305,0.076,0.257,-0.125,1.0
30,7,Lateral_Ventricle,9.2,0.163,0.386,0.537,0.288,0.249,0.284,0.225,1.0
29,8,FA_sCC,11.6,0.135,-0.195,-0.135,-0.254,0.072,0.374,-0.220,1.0
26,9,FA_bCC,11.8,0.141,-0.176,-0.205,-0.148,0.087,0.314,-0.096,1.0
65,10,TotalBrainGMVol_nocereb,12.0,0.131,-0.413,-0.405,-0.425,-0.490,-0.318,-0.688,1.0


,Strategy,Outer folds,Selected per fold,Mean Jaccard,Minimum Jaccard,Maximum Jaccard
0,Control-aware,5,16,0.671,0.524,0.778


## 4. Direct comparison with the FRDA-only baseline

The two strategies use the same folds and select the same number of features. This table therefore isolates the effect of adding control information to the ranking.


In [4]:
recipe_comparison = compare_feature_recipes(baseline_recipe, control_recipe)
summary_row = pd.DataFrame([{
    "Outer fold": "Mean",
    "Baseline features": recipe_comparison["baseline_features"].mean(),
    "Control-aware features": recipe_comparison["control_aware_features"].mean(),
    "Shared features": recipe_comparison["shared_features"].mean(),
    "Jaccard": recipe_comparison["jaccard"].mean(),
    "Removed from baseline": "See fold rows",
    "Added by control-aware": "See fold rows",
}])
comparison_display = recipe_comparison.rename(columns={
    "outer_fold": "Outer fold", "baseline_features": "Baseline features",
    "control_aware_features": "Control-aware features", "shared_features": "Shared features",
    "jaccard": "Jaccard", "removed_from_baseline": "Removed from baseline",
    "added_by_control_aware": "Added by control-aware",
})
display(pd.concat([comparison_display, summary_row], ignore_index=True).round(3))
recipe_comparison.to_csv(SELECTION_DIR / "feature_recipe_comparison.csv", index=False)


,Outer fold,Baseline features,Control-aware features,Shared features,Jaccard,Removed from baseline,Added by control-aware
0,1,16.0,16.0,14.0,0.778,Caudate | RD_SCP,FA_bCC | RD_sCC
1,2,16.0,16.0,12.0,0.600,Caudate | Medulla | Putamen | sCSA_C12_UMN,FA_PCR | Pallidum | RD_PTR | RD_bCC
2,3,16.0,16.0,14.0,0.778,RD_sCC | SCP,FA_ILF_IFOF | Pallidum
3,4,16.0,16.0,13.0,0.684,SCP | TotalBrainWMVol_nocereb | sCSA_C12_UMN,FA_Fx_ST | FA_RLIC | FA_bCC
4,5,16.0,16.0,14.0,0.778,Caudate | FA_RLIC,FA_ILF_IFOF | sCSA_C12_UMN
5,Mean,16.0,16.0,13.4,0.724,See fold rows,See fold rows


## 5. Full-data recipe for final refitting only

This list uses all FRDA and control participants and is therefore not an unbiased performance estimate. It is exported only for final refitting, interpretation, and new-patient scoring after the model comparison is complete.


In [5]:
control_full = build_full_data_feature_recipe(
    frda_long,
    feature_names,
    strategy="control_aware",
    k=FEATURE_COUNT,
    control_long=control_long,
    control_penalty=CONTROL_PENALTY,
)
write_table_artifact(
    SELECTION_DIR / "control_aware_full_data_features.csv",
    control_full,
    schema="feature_recipe",
    manifest=manifest,
    metadata={
        "scope": "full-data deployment only",
        "k": FEATURE_COUNT,
        "control_penalty": CONTROL_PENALTY,
    },
    panel=feature_names,
)
final_display = control_full.loc[control_full["selected"], [
    "rank", "feature", "selection_score", "frda_pooled_d_z", "frda_v1_v2_d_z",
    "frda_v2_v3_d_z", "control_pooled_d_z", "control_v1_v2_d_z", "control_v2_v3_d_z",
]].rename(columns={
    "rank": "Rank", "feature": "Feature", "selection_score": "Selection score",
    "frda_pooled_d_z": "FRDA pooled d", "frda_v1_v2_d_z": "FRDA V1->V2 d",
    "frda_v2_v3_d_z": "FRDA V2->V3 d", "control_pooled_d_z": "Control pooled d",
    "control_v1_v2_d_z": "Control V1->V2 d", "control_v2_v3_d_z": "Control V2->V3 d",
})
display(final_display.round(3))
print(f"Validated fold-specific recipe: {SELECTION_DIR / 'control_aware_features_by_fold.csv'}")
print(f"Deployment-only recipe: {SELECTION_DIR / 'control_aware_full_data_features.csv'}")


,Rank,Feature,Selection score,FRDA pooled d,FRDA V1->V2 d,FRDA V2->V3 d,Control pooled d,Control V1->V2 d,Control V2->V3 d
0,1,Cerebellum_Cortex_CerebNet,0.393,-0.620,-0.878,-0.403,-0.019,-0.169,0.159
1,2,Pons,0.359,-0.552,-0.777,-0.359,0.509,0.411,0.619
2,3,Cerebellum_WM_CerebNet,0.342,-0.425,-0.510,-0.342,0.228,0.319,0.142
3,4,Midbrain,0.324,-0.352,-0.388,-0.324,0.327,0.303,0.349
4,5,Thalamus,0.248,-0.367,-0.469,-0.248,0.091,0.029,0.155
5,6,FA_PTR,0.207,-0.253,-0.207,-0.304,0.076,0.256,-0.125
6,7,Putamen,0.163,-0.339,-0.422,-0.240,-0.155,-0.117,-0.201
7,8,RD_SCP,0.162,0.260,0.162,0.354,-0.085,-0.273,0.082
8,9,Lateral_Ventricle,0.162,0.385,0.536,0.286,0.249,0.283,0.225
9,10,TotalBrainGMVol_nocereb,0.157,-0.413,-0.402,-0.423,-0.490,-0.319,-0.686


Validated fold-specific recipe: /Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/selections/control_aware_features_by_fold.csv
Deployment-only recipe: /Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/selections/control_aware_full_data_features.csv


## Interpretation

This notebook compares feature-selection behaviour, not final biomarker performance. SRM, patient-adaptive, and deep-learning notebooks will apply both fold-specific recipes to held-out FRDA and control participants. No model winner is selected here.
